# 🔥💨 润云 JupyterLab 烟火检测训练 Notebook

同时训练**火检测模型**和**烟检测模型**

## 使用顺序
1. 本地先运行 `convert_voc2yolo_fire.py`，生成 `dataset_fire/`
2. 手动创建只包含smoke标注的数据集（从5张新图片中提取smoke标注）
3. 把 `dataset_fire/` 和 `dataset_smoke_5images_new/` 上传到润云的 `/root/workspace/` 下
4. 按顺序运行下面所有 Cell
5. 训练结束后下载 `fire_yolov8.pt` 和 `smoke_yolov8.pt`


## Cell 1 - 检查环境

In [ ]:
import os
import sys
import subprocess

print('当前工作目录:', os.getcwd())

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip() if result.returncode == 0 else '未检测到 GPU')
print('Python:', sys.version)

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA 可用: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU 设备: {torch.cuda.get_device_name(0)}')
        print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
except Exception as e:
    print('PyTorch 检查失败:', e)


## Cell 2 - 安装依赖

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics 版本:', ultralytics.__version__)


## Cell 3 - 配置路径

In [ ]:
import os

WORKSPACE_DIR = '/root/workspace'

# ============ 火检测数据集 ============
FIRE_DATASET_DIR = os.path.join(WORKSPACE_DIR, 'dataset_fire')
FIRE_YAML_PATH = os.path.join(FIRE_DATASET_DIR, 'dataset_fire.yaml')
FIRE_MODEL_OUTPUT = os.path.join(WORKSPACE_DIR, 'fire_yolov8.pt')

# ============ 烟检测数据集 ============
SMOKE_DATASET_DIR = os.path.join(WORKSPACE_DIR, 'dataset_smoke_5images_new')
SMOKE_YAML_PATH = os.path.join(SMOKE_DATASET_DIR, 'dataset_smoke_5images.yaml')
SMOKE_MODEL_OUTPUT = os.path.join(WORKSPACE_DIR, 'smoke_yolov8.pt')

# ============ 训练输出目录 ============
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, 'runs', 'detect')

print('=' * 50)
print('火检测配置:')
print(f'  DATASET_DIR = {FIRE_DATASET_DIR}')
print(f'  YAML_PATH   = {FIRE_YAML_PATH}')
print(f'  MODEL_OUT   = {FIRE_MODEL_OUTPUT}')
print()
print('烟检测配置:')
print(f'  DATASET_DIR = {SMOKE_DATASET_DIR}')
print(f'  YAML_PATH   = {SMOKE_YAML_PATH}')
print(f'  MODEL_OUT   = {SMOKE_MODEL_OUTPUT}')
print('=' * 50)


## Cell 4 - 检查数据集

In [ ]:
import os
import re

def check_dataset(name, dataset_dir, yaml_path):
    print(f'\n检查 {name} 数据集:')
    if not os.path.exists(dataset_dir):
        print(f'  ❌ 数据集目录不存在: {dataset_dir}')
        return False
    print(f'  ✅ 数据集目录: {dataset_dir}')

    if os.path.exists(yaml_path):
        print(f'  ✅ YAML文件: {yaml_path}')
        # 使用多种编码尝试读取
        yaml_text = None
        for encoding in ['utf-8', 'gbk', 'gb2312', 'latin1']:
            try:
                with open(yaml_path, 'r', encoding=encoding) as f:
                    yaml_text = f.read()
                print(f'  ✅ 使用 {encoding} 编码读取成功')
                break
            except:
                continue
        
        if yaml_text is None:
            print(f'  ❌ 无法读取YAML文件')
            return False
        
        yaml_text = re.sub(r'^path:.*$', f'path: {dataset_dir}', yaml_text, flags=re.MULTILINE)
        with open(yaml_path, 'w', encoding='utf-8') as f:
            f.write(yaml_text)
        print(f'  ✅ YAML路径已修正')
    else:
        print(f'  ❌ YAML文件不存在: {yaml_path}')
        return False

    for split in ['train', 'val']:
        img_dir = os.path.join(dataset_dir, split, 'images')
        lbl_dir = os.path.join(dataset_dir, split, 'labels')
        if os.path.exists(img_dir):
            n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            n_lbl = len([f for f in os.listdir(lbl_dir) if f.lower().endswith('.txt')]) if os.path.exists(lbl_dir) else 0
            print(f'    {split}: {n_img} 张图片, {n_lbl} 个标注')
        else:
            img_dir = os.path.join(dataset_dir, 'images')
            lbl_dir = os.path.join(dataset_dir, 'labels')
            if os.path.exists(img_dir):
                n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                n_lbl = len([f for f in os.listdir(lbl_dir) if f.lower().endswith('.txt')]) if os.path.exists(lbl_dir) else 0
                print(f'    train: {n_img} 张图片, {n_lbl} 个标注')

    return True

# 检查火数据集
fire_ok = check_dataset('火检测', FIRE_DATASET_DIR, FIRE_YAML_PATH)

# 检查烟数据集
smoke_ok = check_dataset('烟检测', SMOKE_DATASET_DIR, SMOKE_YAML_PATH)

if fire_ok and smoke_ok:
    print('\n✅ 所有数据集检查通过!')
else:
    print('\n⚠️ 部分数据集检查失败，请检查路径是否正确')


## Cell 5 - 训练火检测模型

In [ ]:
from ultralytics import YOLO
import torch
import os

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'训练设备: {"GPU: " + torch.cuda.get_device_name(0) if device == 0 else "CPU"}')

print('\n' + '=' * 50)
print('开始训练 火检测模型...')
print('=' * 50)

model = YOLO('yolov8s.pt')

results = model.train(
    data=FIRE_YAML_PATH,
    epochs=150,
    imgsz=768,
    batch=16,
    device=device,
    workers=2,
    project=OUTPUT_DIR,
    name='fire_train',
    exist_ok=True,
    patience=40,
    save=True,
    plots=True,
    augment=True,
    degrees=10,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.3,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    lr0=0.0005,
    lrf=0.01
)

# 复制到固定路径
best_fire_model = os.path.join(OUTPUT_DIR, 'fire_train', 'weights', 'best.pt')
if os.path.exists(best_fire_model):
    os.system(f'cp {best_fire_model} {FIRE_MODEL_OUTPUT}')
    print(f'\n✅ 火模型已保存: {FIRE_MODEL_OUTPUT}')

print('火检测模型训练完成!')


## Cell 6 - 训练烟检测模型

In [ ]:
from ultralytics import YOLO
import torch
import os

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'训练设备: {"GPU: " + torch.cuda.get_device_name(0) if device == 0 else "CPU"}')

print('\n' + '=' * 50)
print('开始训练 烟检测模型...')
print('=' * 50)

model = YOLO('yolov8s.pt')

results = model.train(
    data=SMOKE_YAML_PATH,
    epochs=150,
    imgsz=768,
    batch=16,
    device=device,
    workers=2,
    project=OUTPUT_DIR,
    name='smoke_train',
    exist_ok=True,
    patience=40,
    save=True,
    plots=True,
    augment=True,
    degrees=10,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.3,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    lr0=0.0005,
    lrf=0.01
)

# 复制到固定路径
best_smoke_model = os.path.join(OUTPUT_DIR, 'smoke_train', 'weights', 'best.pt')
if os.path.exists(best_smoke_model):
    os.system(f'cp {best_smoke_model} {SMOKE_MODEL_OUTPUT}')
    print(f'\n✅ 烟模型已保存: {SMOKE_MODEL_OUTPUT}')

print('烟检测模型训练完成!')


## Cell 7 - 查看训练结果

In [ ]:
import os
from IPython.display import Image, display

print('=' * 50)
print('训练结果汇总')
print('=' * 50)

# 火模型
fire_results_plot = os.path.join(OUTPUT_DIR, 'fire_train', 'results.png')
print('\n🔥 火检测模型:')
print(f'  最佳模型: {FIRE_MODEL_OUTPUT}')
print(f'  模型存在: {os.path.exists(FIRE_MODEL_OUTPUT)}')
if os.path.exists(FIRE_MODEL_OUTPUT):
    print(f'  文件大小: {os.path.getsize(FIRE_MODEL_OUTPUT) / 1024 / 1024:.2f} MB')

# 烟模型
smoke_results_plot = os.path.join(OUTPUT_DIR, 'smoke_train', 'results.png')
print('\n💨 烟检测模型:')
print(f'  最佳模型: {SMOKE_MODEL_OUTPUT}')
print(f'  模型存在: {os.path.exists(SMOKE_MODEL_OUTPUT)}')
if os.path.exists(SMOKE_MODEL_OUTPUT):
    print(f'  文件大小: {os.path.getsize(SMOKE_MODEL_OUTPUT) / 1024 / 1024:.2f} MB')

print('\n请在润云文件管理器中下载以上两个模型文件!')


## Cell 8 - 验证火模型

In [ ]:
from ultralytics import YOLO

if os.path.exists(FIRE_MODEL_OUTPUT):
    model = YOLO(FIRE_MODEL_OUTPUT)
    metrics = model.val(data=FIRE_YAML_PATH)
    print(f'\n🔥 火检测模型验证结果:')
    print(f'  mAP@0.5:      {metrics.box.map50:.4f}')
    print(f'  mAP@0.5:0.95: {metrics.box.map:.4f}')
    print(f'  Precision:    {metrics.box.mp:.4f}')
    print(f'  Recall:       {metrics.box.mr:.4f}')
else:
    print('火模型不存在，请先完成训练')


## Cell 9 - 验证烟模型

In [ ]:
from ultralytics import YOLO

if os.path.exists(SMOKE_MODEL_OUTPUT):
    model = YOLO(SMOKE_MODEL_OUTPUT)
    metrics = model.val(data=SMOKE_YAML_PATH)
    print(f'\n💨 烟检测模型验证结果:')
    print(f'  mAP@0.5:      {metrics.box.map50:.4f}')
    print(f'  mAP@0.5:0.95: {metrics.box.map:.4f}')
    print(f'  Precision:    {metrics.box.mp:.4f}')
    print(f'  Recall:       {metrics.box.mr:.4f}')
else:
    print('烟模型不存在，请先完成训练')
